# 07 model training

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
# Load prepared datasets for model training

from pathlib import Path
import pandas as pd

PROJECT_ROOT = PUBLICATION_ROOT

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_DIR = PROJECT_ROOT / "data" / "notebook_outputs" / "ml_ready"

train_df = pd.read_csv(ML_DIR / "train_ml.csv")
val_df = pd.read_csv(ML_DIR / "validation_ml.csv")
test_df = pd.read_csv(ML_DIR / "test_ml.csv")

# SessionID and ground truth are not model inputs
feature_columns = [
    col for col in train_df.columns
    if col not in ["SessionID", "GroundTruthLabel"]
]

# Training data
X_train = train_df[feature_columns]
y_train = train_df["GroundTruthLabel"].astype(int)

# Validation data
X_val = val_df[feature_columns]
y_val = val_df["GroundTruthLabel"].astype(int)

# Testing data, reserved for final evaluation
X_test = test_df[feature_columns]
y_test = test_df["GroundTruthLabel"].astype(int)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nNumber of behavioural features:", len(feature_columns))

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

In [ ]:
# Train the Random Forest baseline model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Initialise Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train using training data ONLY
rf_model.fit(X_train, y_train)

# Predict on validation data
rf_val_predictions = rf_model.predict(X_val)
rf_val_probabilities = rf_model.predict_proba(X_val)[:, 1]

# Evaluate validation performance
print("RANDOM FOREST - VALIDATION RESULTS")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        rf_val_predictions,
        target_names=["Benign", "Malicious"],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_predictions))

print("\nROC-AUC:", round(
    roc_auc_score(y_val, rf_val_probabilities), 4
))

print("Average Precision (PR-AUC summary):", round(
    average_precision_score(y_val, rf_val_probabilities), 4
))

In [ ]:
# Save Random Forest model and validation results

import joblib
import json

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Create output directories
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "random_forest"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# 1. Save trained model
joblib.dump(
    rf_model,
    MODEL_DIR / "random_forest_baseline.joblib"
)


# 2. Calculate and save validation metrics
tn, fp, fn, tp = confusion_matrix(
    y_val, rf_val_predictions, labels=[0, 1]
).ravel()

rf_metrics = {
    "model": "Random Forest",
    "dataset": "validation",
    "training_sessions": len(X_train),
    "validation_sessions": len(X_val),
    "feature_count": len(feature_columns),
    "accuracy": float(accuracy_score(y_val, rf_val_predictions)),
    "precision": float(precision_score(y_val, rf_val_predictions)),
    "recall": float(recall_score(y_val, rf_val_predictions)),
    "f1_score": float(f1_score(y_val, rf_val_predictions)),
    "roc_auc": float(roc_auc_score(y_val, rf_val_probabilities)),
    "average_precision": float(
        average_precision_score(y_val, rf_val_probabilities)
    ),
    "confusion_matrix": {
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    },
    "feature_columns": feature_columns
}

with open(RESULTS_DIR / "validation_metrics.json", "w") as f:
    json.dump(rf_metrics, f, indent=4)


# 3. Save individual validation predictions
rf_predictions_df = pd.DataFrame({
    "SessionID": val_df["SessionID"].values,
    "ActualLabel": y_val.values,
    "PredictedLabel": rf_val_predictions,
    "MaliciousProbability": rf_val_probabilities
})

rf_predictions_df.to_csv(
    RESULTS_DIR / "validation_predictions.csv",
    index=False
)


# 4. Save feature importance
rf_feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

rf_feature_importance.to_csv(
    RESULTS_DIR / "feature_importance.csv",
    index=False
)


print("Random Forest model and results saved successfully.")
print("\nValidation metrics:")
display(pd.DataFrame([rf_metrics]).drop(
    columns=["confusion_matrix", "feature_columns"]
))

print("\nTop 10 important features:")
display(rf_feature_importance.head(10))

In [ ]:
import matplotlib.pyplot as plt

top_10 = rf_feature_importance.sort_values("Importance", ascending=False).head(10)
top_10 = top_10.sort_values("Importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_10["Feature"], top_10["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Random Forest Feature Importances")
plt.tight_layout()

plot_path = PROJECT_ROOT / "results" / "random_forest" / "feature_importance_plot.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", plot_path)

In [ ]:
print("Training features:", X_train.columns.tolist())

assert "GroundTruthLabel" not in X_train.columns
assert "GroundTruthLabel" not in X_val.columns
assert "GroundTruthLabel" not in X_test.columns

print("\nNo ground-truth labels are present in model inputs.")

Install dependencies before starting the kernel, using the pinned requirements files.
`python -m pip install -r requirements-ml.txt`
Run that command from the repository root. This notebook does not install or upgrade packages.


In [ ]:
import os
from getpass import getpass

os.environ["TABPFN_TOKEN"] = getpass(
    "Enter your Prior Labs API key: "
)

print("TabPFN authentication token configured.")

In [ ]:
# Train TabPFN and evaluate on validation data

from tabpfn import TabPFNClassifier
from importlib.metadata import version

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("TabPFN package version:", version("tabpfn"))

# Initialise TabPFN
tabpfn_model = TabPFNClassifier(
    device="auto",
    random_state=42
)

# Fit using training data only
tabpfn_model.fit(X_train, y_train)

# Predict on validation data
tabpfn_val_predictions = tabpfn_model.predict(X_val)

tabpfn_val_probabilities = tabpfn_model.predict_proba(X_val)[
    :, list(tabpfn_model.classes_).index(1)
]

# Evaluate validation performance
print("\nTABPFN - VALIDATION RESULTS")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        tabpfn_val_predictions,
        target_names=["Benign", "Malicious"],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, tabpfn_val_predictions))

print("\nROC-AUC:", round(
    roc_auc_score(y_val, tabpfn_val_probabilities), 4
))

print("Average Precision:", round(
    average_precision_score(y_val, tabpfn_val_probabilities), 4
))

In [ ]:
# Save TabPFN fitted model and validation results

import json
from importlib.metadata import version
from tabpfn.model_loading import save_fitted_tabpfn_model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

TABPFN_RESULTS_DIR = PROJECT_ROOT / "results" / "tabpfn"
TABPFN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save the fitted TabPFN model
save_fitted_tabpfn_model(
    tabpfn_model,
    str(MODEL_DIR / "tabpfn_baseline.tabpfn_fit")
)

# 2. Calculate validation metrics
tn, fp, fn, tp = confusion_matrix(
    y_val, tabpfn_val_predictions, labels=[0, 1]
).ravel()

tabpfn_metrics = {
    "model": "TabPFN",
    "package_version": version("tabpfn"),
    "dataset": "validation",
    "training_sessions": len(X_train),
    "validation_sessions": len(X_val),
    "feature_count": len(feature_columns),
    "accuracy": float(accuracy_score(y_val, tabpfn_val_predictions)),
    "precision": float(precision_score(y_val, tabpfn_val_predictions)),
    "recall": float(recall_score(y_val, tabpfn_val_predictions)),
    "f1_score": float(f1_score(y_val, tabpfn_val_predictions)),
    "roc_auc": float(roc_auc_score(y_val, tabpfn_val_probabilities)),
    "average_precision": float(
        average_precision_score(y_val, tabpfn_val_probabilities)
    ),
    "confusion_matrix": {
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    },
    "feature_columns": feature_columns
}

with open(TABPFN_RESULTS_DIR / "validation_metrics.json", "w") as f:
    json.dump(tabpfn_metrics, f, indent=4)

# 3. Save validation predictions
tabpfn_predictions_df = pd.DataFrame({
    "SessionID": val_df["SessionID"].values,
    "ActualLabel": y_val.values,
    "PredictedLabel": tabpfn_val_predictions,
    "MaliciousProbability": tabpfn_val_probabilities
})

tabpfn_predictions_df.to_csv(
    TABPFN_RESULTS_DIR / "validation_predictions.csv",
    index=False
)

print("TabPFN model and results saved successfully.")

display(
    pd.DataFrame([tabpfn_metrics]).drop(
        columns=["confusion_matrix", "feature_columns"]
    )
)

In [ ]:
# Compare Random Forest and TabPFN validation results

import json
import pandas as pd

# Load saved validation metrics
with open(PROJECT_ROOT / "results/random_forest/validation_metrics.json") as f:
    rf_results = json.load(f)

with open(PROJECT_ROOT / "results/tabpfn/validation_metrics.json") as f:
    tabpfn_results = json.load(f)

# Create comparison table
metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "average_precision"
]

comparison = pd.DataFrame(
    [rf_results, tabpfn_results]
)[["model"] + metrics].set_index("model")

print("MODEL VALIDATION COMPARISON")
display(comparison.round(4))

# Save comparison
comparison.to_csv(
    PROJECT_ROOT / "results/model_validation_comparison.csv"
)

print("\nComparison saved successfully.")